# 预测到客流后，应该安排多少工作人员？：参考答案

每道题先写预期结果再运行。练习中的 TODO 需要自行完成；能顺序运行不代表已完成练习。

In [1]:
from pathlib import Path
CASE_DIR = Path.cwd()
# 载入本例的可修改函数和数据，所有算法都在同目录 solution.py。
exec(compile((CASE_DIR / "solution.py").read_text(encoding="utf-8"), str(CASE_DIR / "solution.py"), "exec"))

{'selected_on_validation': 'mean', 'validation': {'fixed': {'cost': 179.24, 'mean_staff': 5.0, 'shortage_rate': 0.205, 'mean_shortage': 3.655}, 'linear_mean': {'cost': 173.04, 'mean_staff': 4.74, 'shortage_rate': 0.275, 'mean_shortage': 3.855}, 'mean': {'cost': 152.36, 'mean_staff': 4.72, 'shortage_rate': 0.18, 'mean_shortage': 1.345}, 'buffered': {'cost': 165.43, 'mean_staff': 5.485, 'shortage_rate': 0.02, 'mean_shortage': 0.11}}, 'test': {'fixed': {'cost': 181.712, 'mean_staff': 5.0, 'shortage_rate': 0.276, 'mean_shortage': 3.964}, 'linear_mean': {'cost': 177.608, 'mean_staff': 4.684, 'shortage_rate': 0.34, 'mean_shortage': 4.636}, 'mean': {'cost': 152.128, 'mean_staff': 4.72, 'shortage_rate': 0.22, 'mean_shortage': 1.316}, 'buffered': {'cost': 164.392, 'mean_staff': 5.468, 'shortage_rate': 0.016, 'mean_shortage': 0.044}}, 'model_comparison': [{'model': 'ridge', 'validation_mae': 14.07781932436093, 'test_mae': 16.11330750501616, 'validation_decision_cost': 173.04, 'test_decision_cost

## 1. 补全：容量与缺口
手工计算选中方案的平均缺口，并与输出指标核对。

In [2]:
gap=np.maximum(y_test-test_staff[selected]*capacity,0)
assert np.isclose(gap.mean(),test_scores[selected]['mean_shortage'])
print(gap.mean())

1.316


## 2. 修改：缺口成本翻倍
用验证集客流把缺口成本改为 16，重新比较策略；不需要重新训练预测模型。

In [3]:
new_cost={name:float(np.mean(staff*wage+np.maximum(y_val-staff*capacity,0)*16)) for name,staff in staffing_predictions(X_val).items()}
assert all(np.isfinite(list(new_cost.values())))
print(new_cost,'新方案',min(new_cost,key=new_cost.get))

{'fixed': 208.48, 'linear_mean': 203.88, 'mean': 163.12, 'buffered': 166.31} 新方案 mean


## 3. 迁移：人员最多 6 人
给测试时段配置加入 6 人上限，输出被截断时段数与新缺口率。

In [4]:
capped=np.minimum(test_staff[selected],6)
assert capped.max()<=6
print({'截断时段':int(np.sum(test_staff[selected]>6)),**evaluate_staff(capped,y_test)})

{'截断时段': 9, 'cost': 153.672, 'mean_staff': 4.684, 'shortage_rate': 0.236, 'mean_shortage': 1.644}
